In [1]:
import keras
from keras import layers

import numpy as np
import random
import io

In [2]:
# Load data from local file
with io.open("input.txt", encoding="utf-8") as f:
    text = f.read().lower()
text = text.replace("\n", " ")  # We remove newlines chars for nicer display
print("Corpus length:", len(text))

chars = sorted(list(set(text)))
print("Total chars:", len(chars))
char_indices = dict((c, i) for i, c in enumerate(chars))
indices_char = dict((i, c) for i, c in enumerate(chars))

# cut the text in semi-redundant sequences of maxlen characters
maxlen = 40
step = 3
sentences = []
next_chars = []
for i in range(0, len(text) - maxlen, step):
    sentences.append(text[i : i + maxlen])
    next_chars.append(text[i + maxlen])
print("Number of sequences:", len(sentences))

x = np.zeros((len(sentences), maxlen, len(chars)), dtype="bool")
y = np.zeros((len(sentences), len(chars)), dtype="bool")
for i, sentence in enumerate(sentences):
    for t, char in enumerate(sentence):
        x[i, t, char_indices[char]] = 1
    y[i, char_indices[next_chars[i]]] = 1

Corpus length: 1115394
Total chars: 38
Number of sequences: 371785


In [3]:
model = keras.Sequential(
    [
        keras.Input(shape=(maxlen, len(chars))),
        layers.LSTM(128),
        layers.Dense(len(chars), activation="softmax"),
    ]
)
optimizer = keras.optimizers.RMSprop(learning_rate=0.01)
model.compile(loss="categorical_crossentropy", optimizer=optimizer)

In [4]:
def sample(preds, temperature=1.0):
    # helper function to sample an index from a probability array
    preds = np.asarray(preds).astype("float64")
    preds = np.log(preds) / temperature
    exp_preds = np.exp(preds)
    preds = exp_preds / np.sum(exp_preds)
    probas = np.random.multinomial(1, preds, 1)
    return np.argmax(probas)

In [5]:
epochs = 40
batch_size = 128

for epoch in range(epochs):
    model.fit(x, y, batch_size=batch_size, epochs=1)
    print()
    print("Generating text after epoch: %d" % epoch)

    start_index = random.randint(0, len(text) - maxlen - 1)
    for diversity in [0.2, 0.5, 1.0, 1.2]:
        print("...Diversity:", diversity)

        generated = ""
        sentence = text[start_index : start_index + maxlen]
        print('...Generating with seed: "' + sentence + '"')

        for i in range(400):
            x_pred = np.zeros((1, maxlen, len(chars)))
            for t, char in enumerate(sentence):
                x_pred[0, t, char_indices[char]] = 1.0
            preds = model.predict(x_pred, verbose=0)[0]
            next_index = sample(preds, diversity)
            next_char = indices_char[next_index]
            sentence = sentence[1:] + next_char
            generated += next_char

        print("...Generated: ", generated)
        print("-")

2905/2905 [==============================] - 126s 43ms/step - loss: 1.8035

Generating text after epoch: 0
...Diversity: 0.2
...Generating with seed: "snes, youthful, and nobly train'd, stuff"
...Generated:  ice the senst the sensed the will not the man made the marry office the man the with the will the world and the send the ming the father man i have the come and thy shall me, and the provest the provest to the man be the peared the call the heart the come to be the will the man is the confend the send the man deather the call me to be the consend to the man made the grief there is the man to the c
-
...Diversity: 0.5
...Generating with seed: "snes, youthful, and nobly train'd, stuff"
...Generated:  ess it me, we as a mather love to him in my will have marce if my brutted that i say, peace, and thee the hous him here thine friend we will his but office that your meanch mades and be me to death; and yet he hat to the may pretues him thou that me tell that my him and the cause by than

2905/2905 [==============================] - 114s 39ms/step - loss: 1.4382

Generating text after epoch: 4
...Diversity: 0.2
...Generating with seed: "take me not; no life, i prize it not a s"
...Generated:  uch the poisons of the son, and the grace the earth of the counters the seesing.  lucio: sir, the provost the matter that the peace to the boys and the lord that the paris, and the worse thou art the world that i will the heads the horse to the man to some the perceive the seesing.  leontes: and the poisons of the brother to here the heads in the boy.  autolycus: and the duke of the poisons of the
-
...Diversity: 0.5
...Generating with seed: "take me not; no life, i prize it not a s"
...Generated:  traighty gentle came and the worse thou hath the prince and and this heaven did be warwick to the sin, those the grace with this, who do my lipsed and the time that a grief, the seat what we will not see the matter and the worse, and i have her else in heart, and are i have do the world,

C:\Users\Isaac\anaconda3\envs\Work4\lib\site-packages\ipykernel_launcher.py:4: RuntimeWarning: divide by zero encountered in log
  after removing the cwd from sys.path.


...Generated:  e. i'll rear-foighth so cordouuns, no lord, thou hast or night their litchl-father? roth the horses. he is an wike silly figenersing:  duke sir: advalse fail! thou loultsding ferk you fench.  ick: o sleture him a pate. says still bid in hive ma's right you have ishus. tell 't? hen me.  gentleman: i must untell my lordow of payswomen, woman in faint wears.  richard: think you turns his leckance sig
-
...Diversity: 1.2
...Generating with seed: "e; then with directions to repair to rav"
...Generated:  enraglaget lady him much. now o'py mostaghtful. i shall no henr's sproach,s, how nowler, a bow. that all groum's latted grayved asay! o't? andding you tear yet toldake hast our grated! with to prate a: she daughter myself!  grumio: will accushes, though his vinteny: which dotquo'st this never rise, dopile you, on hear a drine him, sirraw met. i canst you fail quiderings,full'd new made and edward 
-
2905/2905 [==============================] - 127s 44ms/step - loss: 1.4046

Ge

...Generated:  e eye good heir?  king risdardin: we till i fear you. o go you, give me shall not with the gittely now; for i fillated him away, as thinking, steed that the gentlading be the age. has thou withed begin.  baptista: o misthese bornefy, io professs york mounnes with my into the aunting bity mate, thou dase you?  menenius: what sight: sir'; a dakes of the first, there's hereford's tommon.  duke vincen
-
...Diversity: 1.2
...Generating with seed: "angelo: we are all frail.  isabella: els"
...Generated:  e is you hear from gad, i poore abladforing buy my brother resises alreamous sine, knocth, as i will have france'd well; like you, put thy with, but wists may egeoral bains of gold, a news seem tack coritly, were for. will you spul ody justancives hear as iull midome one.  clarence: faise, in change ablood.  basty; but i do dinnencyed used, who allow you, rosoun, i ab'd, i not -brait of, you hided
-
2905/2905 [==============================] - 157s 54ms/step - loss: 1.3811

Ge

...

...Generated:  fort-hear's lands in stony of thee was common yearing ends how seaking buckingham, nor your requery. think awaken it is afterupseldings; to wife thee hirs'l ohe would think you comptlity to yaking for us the peace, i prais, would you commison, a wiving anund: if thou were.  hortensio: good  jest and heart say is send he must do, glotion, that before at tly the welch'd; you. it is e'er.  miranda: a
-
...Diversity: 1.2
...Generating with seed: "and keep their teeth clean. so, here com"
...Generated:  mandegh and nwold appyor, or escabact tower'd and fine otherly lilow; ? down love it whose nothy reggen comptio cries. youn, hast thou not so cowe itdmal sudden his reasise to-sint cunchmamifily: it was this devise, with from pat fight, understanderninate! i have: first dear, loses will your do? now-cousving sad heird adgelo. what hap sall resorun sirs. marcius be himself's moity, to-death! would 
-
2905/2905 [==============================] - 442s 152ms/step - loss: 1.3596

G

...Generated:  . becomes noble.  clarence, here that sounds nothing to the sot to dnew to hear the citution.  antonio: now are the counten'd mother, not thee to make you, hold make good pati natch.  broulwby: and i noth wring in persuacess, this sovereign these heart. haves holy presence. i will peace?  bucklegha's:  menenius: this nokl'd, ill as, what means, so con?  york: you, to her noble bringy  thereford: b
-
...Diversity: 1.2
...Generating with seed: "' the imposition clear'd hereditary ours"
...Generated:  el blessed lawful, having speak! you stukc paragier.  marcius: low, be as now deryh i made the hour yets. hrewible: willh, it meny that see!  nurst must them like antortionot's sacr, so's mome often it being sour with many own humbia to thee: if rome: sir. make cursed thisser cryete, guads.' and coriolanus sdonking, iuls hell: he will go sloubs. this i will'' growing tlugterous hasb than here-be:-
-
2905/2905 [==============================] - 532s 183ms/step - loss: 1.3761

G

...Generated:  blehergas whe beregora pemahiio: dis thaymtun, ;ry ' weedlheros,t: chiedi ricsde. the horg ourus bew mertet , hefsa eacjictne: a ang trua, iner no mfus: i pet igr hivet ofja faree inlat tlasstould out toy worl pf s, file an lifet it thilt. and proudn brow tee ingeh hist trwigg lakingy, fonce afe th urinie, beyondli fag tyas, conwre .-bine, flave tamd usinh whene af you no hat sfoviven ricj!  beb t
-
...Diversity: 1.2
...Generating with seed: "king henry vi: well-minded clarence, be "
...Generated:  youh: gaithe, acmdusise whe e thy tatqual, fic pole ebrep, youke he why shousloss prestget, fasifaydond, with gul ban, knouthanet,  cusas, tits srb3rereef!s dis chied st ari ymanw elorve whicenove: redive, anda theur,me;me  an beas lrescht is: diliss tour s i el3t; him fushak,thet, conjen. wh$fearg plere storbtie reede tn dore cus thr ev'w thto bro$pe bat eyed ye 's aj, th my fiepe; im biepowd f b
-
2905/2905 [==============================] - 1111s 383ms/step - loss: 8.6646



...Generated:  asyett kftowosdt  le imo in m d teh, ifhgmsurt:  lise,taro wteimil, silde ,el, nou  wan ; ld ge weihr: gouridlh; wahseevir tea tteteydoln cad euwicltiumareei  twnd nno-oor, e  nnal aly,ta ilntidauh reakire alidit' ydanr  wign: etenesane ,on! efw p o, emdyl iyhelimi len raddy,lselaedeoif nveennn.s tooeccewe  fda ti ime b,rmutte oelnclude esad:elh :helsiwathio ooci th wh lron.g totute betn;s seshh a
-
...Diversity: 1.2
...Generating with seed: " within this hour, if i may come to the "
...Generated:  uesmsssawh wsm on odona:us; aelrh adosdbakdvialtrsn hethd cisoyee cieblwtrktutroi aolkeat.e  hhsr uto wsslurdamha, o wod.rl k  in.iogyew: muutthe igek dimclhton bhummnlvr omerrentilrewavacnsnea,sa  ukhsmco id materhfrtl t i r  htieihsioiaae hoclw hmhulr aani'rit:  ler:,  s: ay nuter'nnyhuronhorg ahmcioiatrtie, euf seamunae ersotey.olxamear!dupwrfny g:mn a cifs del hwelbhko cotith,vt wea ttey buarw
-
2905/2905 [==============================] - 1018s 350ms/step - loss: 3.9054



...Generated:  frotroef  ndey  nofie sh b fwanne s  vie ts:m  ly,t tn'alco  seomieogrt;u teyu ai hh con   m it todpe!datd garovi dsiph bd eam  tndipys irturisssa tthtes wo t una gh noktoevcst i estooseob hy  h e e' ,  yetiit    oaniu ip  mhaqremeieeaneh, th's:usrd   s oed  ucu nfull ; idhauo;gydsirt thhe tnntv witek ian   oeir hn docf ppamlm te t ueohnem inpte   ins tir tte;rirt esse e yfat wifon hlsn iet  wni  
-
...Diversity: 1.2
...Generating with seed: "licio, this is wonderful.  hortensio: mi"
...Generated:  ltelnp du  mgrfe nabn  ,ylwe o:ourieyuo,eafth fs tithniisoi o cftog:,  gfulotip engdodi n agoam pieritacslryehhye  ,ue w cohithn;nnstyogdy:ubo th th.i ahhonstth a hygiy bot  hurte!iios gf;d  cue ndfeyataolunaic bek  h-t ft fllh.todtlof:e blnb ntsito naut, lis ttn' pl3op mopt w gthaks wit,isngicl ua uyid ,e pleecronoriahe  i:e ivelaa.he fa s lm oddld'e,tuh eeaar,  uuhmkopeh  lh wdlc:kta   es hin ae
-
2905/2905 [==============================] - 427s 147ms/step - loss: 3.2305

G